# 04 — Cribado por título y resumen

Este notebook organiza el primer cribado de elegibilidad del mapa mundial de evidencia sobre cambio climático y pesquerías marinas, con prioridad analítica para pequeños pelágicos y la anchoveta peruana.

La decisión se basa únicamente en **título, resumen y metadatos disponibles**. Se evalúan por separado tres criterios:

1. relevancia para pesquerías o sistemas marinos;
2. presencia de un componente climático, ambiental, ecosistémico, precautorio, de adaptación o resiliencia;
3. existencia de al menos un aporte codificable.

Regla operativa:

- `include`: los tres criterios son `yes`;
- `exclude`: al menos un criterio es claramente `no` y se registra una razón;
- `uncertain`: al menos un criterio es `unclear`; el registro avanza a aclaración o texto completo.

Los términos diagnósticos solo priorizan el orden de revisión. **No deciden automáticamente la inclusión o exclusión.**


In [ ]:
from pathlib import Path
import sys

import pandas as pd

ROOT = Path.cwd().resolve()
if ROOT.name == "notebooks":
    ROOT = ROOT.parent

SRC = ROOT / "src"
if str(SRC) not in sys.path:
    sys.path.insert(0, str(SRC))

from evidence_review.screening import (
    apply_keyword_triage,
    build_screening_corpus,
    completed_binary_decisions,
    export_prompt_jsonl,
    initialise_screening_sheet,
    load_screening_config,
    screening_summary,
    select_pilot_sample,
    validate_screening_sheet,
)

INTERIM = ROOT / "data" / "interim"
INTERIM.mkdir(parents=True, exist_ok=True)

print(f"Project root: {ROOT}")


## 1. Cargar reglas de cribado

Las definiciones, códigos de exclusión y grupos prioritarios se mantienen fuera del notebook para que sean versionables y auditables.


In [ ]:
config_path = ROOT / "config" / "screening.yml"
config = load_screening_config(config_path)

print("Stage:", config["review_stage"])
print("Decisions:", ", ".join(config["decisions"]))
print("Criterion values:", ", ".join(config["criterion_values"]))
print("Exclusion reasons:", len(config["exclusion_reasons"]))
print("Priority groups:", len(config["priority_groups"]))


## 2. Leer el corpus deduplicado

El archivo de entrada proviene del notebook 03. Se conserva el resumen porque el registro maestro bibliográfico no lo almacena como campo principal.


In [ ]:
retained_path = INTERIM / "deduplicated_sources.csv"

if not retained_path.exists():
    raise FileNotFoundError(
        "No existe data/interim/deduplicated_sources.csv. "
        "Ejecuta primero notebooks/03_import_and_deduplicate_sources.ipynb."
    )

retained = pd.read_csv(retained_path).fillna("")

if retained.empty:
    raise ValueError(
        "El corpus deduplicado está vacío. Importa al menos una exportación "
        "bibliográfica antes de iniciar el cribado."
    )

print(f"Retained sources: {len(retained)}")
print(f"Unique titles: {retained['title'].nunique() if 'title' in retained else 0}")


## 3. Construir el corpus de título y resumen

Se generan identificadores estables compatibles con el registro maestro y diagnósticos de disponibilidad del resumen.


In [ ]:
screening_corpus = build_screening_corpus(retained)
screening_corpus = apply_keyword_triage(screening_corpus, config)

corpus_path = INTERIM / "title_abstract_screening_corpus.csv"
screening_corpus.to_csv(corpus_path, index=False, encoding="utf-8-sig")

print(f"Sources prepared: {len(screening_corpus)}")
print(f"Abstract available: {int(screening_corpus['abstract_available'].sum())}")
print(f"Abstract missing/short: {int((~screening_corpus['abstract_available']).sum())}")
print(f"Saved: {corpus_path.relative_to(ROOT)}")

display(
    screening_corpus[
        [
            "source_id",
            "title",
            "year",
            "abstract_available",
            "triage_priority",
            "suggested_priority_groups",
        ]
    ].head(10)
)


## 4. Revisar el triaje diagnóstico

`high` indica coincidencia entre clima/ambiente y pequeños pelágicos o peces forrajeros. `medium` indica clima/ambiente y pesquerías marinas. La ausencia de palabras clave no demuestra irrelevancia.


In [ ]:
triage_summary = (
    screening_corpus["triage_priority"]
    .value_counts(dropna=False)
    .rename_axis("triage_priority")
    .reset_index(name="sources")
)

display(triage_summary)

display(
    screening_corpus.sort_values(
        ["triage_priority", "abstract_available"],
        ascending=[True, False],
    )[
        [
            "source_id",
            "title",
            "triage_priority",
            "climate_keyword_hits",
            "fisheries_keyword_hits",
            "priority_taxa_keyword_hits",
            "exclusion_keyword_hits",
        ]
    ].head(20)
)


## 5. Crear o actualizar la hoja de trabajo

La hoja se reconstruye desde el corpus actual, pero conserva las decisiones manuales existentes mediante `source_id`. Así se pueden incorporar nuevas exportaciones sin perder el trabajo previo.


In [ ]:
working_path = INTERIM / "title_abstract_screening_working.csv"

if working_path.exists():
    existing_screening = pd.read_csv(working_path).fillna("")
    print(f"Existing screening rows: {len(existing_screening)}")
else:
    existing_screening = pd.DataFrame()
    print("No previous screening sheet found.")

screening_sheet = initialise_screening_sheet(
    screening_corpus,
    existing=existing_screening,
)

screening_sheet.to_csv(working_path, index=False, encoding="utf-8-sig")

print(f"Working rows: {len(screening_sheet)}")
print(f"Saved: {working_path.relative_to(ROOT)}")
display(screening_summary(screening_sheet))


## 6. Libro de códigos para la revisión manual

Valores permitidos para cada criterio:

- `yes`: el título o resumen lo respalda explícitamente;
- `no`: el título o resumen permite rechazar el criterio con claridad;
- `unclear`: información ausente, truncada o ambigua.

No se excluye una fuente solo porque no mencione pequeños pelágicos. Ese grupo es prioritario, pero el alcance general incluye evidencia transferible de otras pesquerías marinas.


In [ ]:
criteria_table = pd.DataFrame(
    [
        {
            "criterion": name,
            "question": details["question"],
        }
        for name, details in config["eligibility_criteria"].items()
    ]
)

exclusion_table = pd.DataFrame(
    {"allowed_exclusion_reason": config["exclusion_reasons"]}
)
priority_table = pd.DataFrame(
    {"allowed_priority_group": config["priority_groups"]}
)

display(criteria_table)
display(exclusion_table)
display(priority_table)


## 7. Seleccionar una muestra piloto reproducible

Para un corpus de hasta 50 fuentes se revisan todas. Para corpus mayores se genera inicialmente una muestra de 30 fuentes distribuida entre estratos de prioridad diagnóstica.


In [ ]:
pilot_n = min(30, len(screening_sheet))
pilot_sample = select_pilot_sample(
    screening_sheet,
    n=pilot_n,
    random_state=42,
)

pilot_path = INTERIM / "title_abstract_screening_pilot.csv"
pilot_sample.to_csv(pilot_path, index=False, encoding="utf-8-sig")

print(f"Pilot sources: {len(pilot_sample)}")
print(f"Saved: {pilot_path.relative_to(ROOT)}")
display(
    pilot_sample[
        [
            "source_id",
            "title",
            "abstract",
            "triage_priority",
            "suggested_priority_groups",
            "decision",
        ]
    ]
)


## 8. Preparar prompts auditables para asistencia opcional

Esta celda **no llama a ninguna API**. Solo genera un archivo JSONL con un prompt por fuente. La decisión de un modelo debe considerarse una propuesta y requiere validación humana.


In [ ]:
prompt_path = INTERIM / "title_abstract_screening_prompts.jsonl"
export_prompt_jsonl(pilot_sample, config, prompt_path)

print(f"Prompt records: {len(pilot_sample)}")
print(f"Saved: {prompt_path.relative_to(ROOT)}")


## 9. Validar decisiones registradas

Edita `data/interim/title_abstract_screening_working.csv` en Excel, LibreOffice o VS Code, guárdalo y vuelve a ejecutar desde esta celda. Las filas aún no revisadas pueden permanecer vacías.


In [ ]:
reviewed_sheet = pd.read_csv(working_path).fillna("")
validation_issues = validate_screening_sheet(reviewed_sheet, config)

issues_path = INTERIM / "title_abstract_screening_issues.csv"
validation_issues.to_csv(issues_path, index=False, encoding="utf-8-sig")

print(f"Validation issues: {len(validation_issues)}")
print(f"Saved: {issues_path.relative_to(ROOT)}")
display(validation_issues.head(50))


## 10. Resumen del cribado y exportación binaria

Las decisiones `uncertain` permanecen en la hoja de trabajo y avanzan a recuperación de texto completo. Solo las filas válidas `include` o `exclude` se convierten al esquema binario general del proyecto.


In [ ]:
progress = screening_summary(reviewed_sheet)
display(progress)

if validation_issues.empty:
    binary_decisions = completed_binary_decisions(reviewed_sheet)
else:
    invalid_ids = set(validation_issues["source_id"].dropna().astype(str))
    valid_rows = reviewed_sheet.loc[
        ~reviewed_sheet["source_id"].astype(str).isin(invalid_ids)
    ].copy()
    binary_decisions = completed_binary_decisions(valid_rows)

binary_path = INTERIM / "screening_decisions_working.csv"
binary_decisions.to_csv(binary_path, index=False, encoding="utf-8-sig")

included_path = INTERIM / "title_abstract_included.csv"
uncertain_path = INTERIM / "title_abstract_uncertain.csv"
excluded_path = INTERIM / "title_abstract_excluded.csv"

reviewed_sheet.loc[reviewed_sheet["decision"] == "include"].to_csv(
    included_path, index=False, encoding="utf-8-sig"
)
reviewed_sheet.loc[reviewed_sheet["decision"] == "uncertain"].to_csv(
    uncertain_path, index=False, encoding="utf-8-sig"
)
reviewed_sheet.loc[reviewed_sheet["decision"] == "exclude"].to_csv(
    excluded_path, index=False, encoding="utf-8-sig"
)

print(f"Binary decisions exported: {len(binary_decisions)}")
print(f"Included: {(reviewed_sheet['decision'] == 'include').sum()}")
print(f"Uncertain: {(reviewed_sheet['decision'] == 'uncertain').sum()}")
print(f"Excluded: {(reviewed_sheet['decision'] == 'exclude').sum()}")


## 11. Control de razones de exclusión

Las razones deben reflejar el primer criterio claramente incumplido. `insufficient_metadata` no se usa como exclusión automática en esta etapa; cuando falta información sustantiva se utiliza `uncertain`.


In [ ]:
excluded = reviewed_sheet.loc[reviewed_sheet["decision"] == "exclude"]

if excluded.empty:
    exclusion_summary = pd.DataFrame(
        columns=["exclusion_reason", "sources"]
    )
else:
    exclusion_summary = (
        excluded["exclusion_reason"]
        .value_counts(dropna=False)
        .rename_axis("exclusion_reason")
        .reset_index(name="sources")
    )

display(exclusion_summary)


## Criterio para avanzar

La fase de texto completo puede comenzar cuando:

- se haya completado y validado el piloto;
- todas las decisiones tengan tres criterios codificados;
- cada exclusión tenga una razón válida;
- las fuentes `uncertain` estén identificadas para aclaración o recuperación de texto completo;
- la sensibilidad de la búsqueda sea aceptable para anchoveta, otros pequeños pelágicos y evidencia transferible;
- se haya documentado cualquier cambio necesario en la estrategia de búsqueda.

La siguiente fase debe separar dos rutas:

1. recuperación y evaluación de texto completo;
2. descarga, parsing y segmentación documental para las fuentes incluidas o inciertas.
